In [3]:
import os
import re
import pandas as pd
import subprocess
import py3Dmol
from glob import glob
from tqdm.auto import tqdm

results_dirs = glob("../data_files/output/diffdock/index*")
print(results_dirs)

rows = []
for results_dir in tqdm(results_dirs, desc="runs"):
    results_pdb_file = "../data_files/inference/pdb/" + re.findall("inference-pdb-(.+\.pdb)", results_dir)[0]
    results_smiles = re.findall("pdb-+(.+)", results_dir)[0]
    results_sdfs = [os.path.join(results_dir, f) for f in os.listdir(results_dir) if "confidence" in f and f.endswith(".sdf")]
    results_pdb_file_no_hetatms = f"{results_pdb_file}_nohet.pdb"

    # grep -v "^HETATM" {results_pdb_file} > {results_pdb_file_no_hetatms}
    # cp {results_pdb_file} .
    with open(results_pdb_file, 'r') as infile:  
        lines = infile.readlines()  

    with open(results_pdb_file_no_hetatms, 'w') as outfile:  
        for line in lines:  
            if not line.startswith('HETATM'):  
                outfile.write(line)  

    for results_sdf in tqdm(results_sdfs, leave=False, desc="files"):
        confidence = re.findall("confidence([\-\.\d]+)\.sdf", results_sdf)[0]
        # scored_stdout = !/content/DiffDock/gnina --score_only -r "{results_pdb_file_no_hetatms}" -l "{results_sdf}"
        command = f"./gnina --score_only -r \"{results_pdb_file_no_hetatms}\" -l \"{results_sdf}\""
        result = subprocess.run(command, shell=True, stdout=subprocess.PIPE)
        scored_stdout = result.stdout.decode()
        scored_affinity = re.findall("Affinity:\s*([\-\.\d+]+)", scored_stdout)[0]
        
        command = f"./gnina --local_only --minimize -r \"{results_pdb_file_no_hetatms}\" -l \"{results_sdf}\" --autobox_ligand \"{results_sdf}\" --autobox_add 2"
        result = subprocess.run(command, shell=True, stdout=subprocess.PIPE)
        minimized_stdout = result.stdout.decode()
        minimized_affinity = re.findall("Affinity:\s*([\-\.\d+]+)", minimized_stdout)[0]
        rows.append((results_pdb_file, results_smiles, float(confidence), float(scored_affinity), float(minimized_affinity), results_sdf))
        
# create dataframe, tar file and download
df_results = pd.DataFrame(rows, columns=["pdb_file", "smiles", "diffdock_confidence", "gnina_scored_affinity", "gnina_minimized_affinity", "sdf_file"])
df_results_tsv = "df_diffdock_results.tsv"
df_results.to_csv(df_results_tsv, sep='\t', index=None)

resid_hover = """
function(atom,viewer) {
    if(!atom.label) {
        atom.label = viewer.addLabel(atom.chain+" "+atom.resn+" "+atom.resi,
            {position: atom, backgroundColor: 'mintcream', fontColor:'black', fontSize:12});
    }
}"""
unhover_func = """
function(atom,viewer) {
    if(atom.label) {
        viewer.removeLabel(atom.label);
        delete atom.label;
    }
}"""

view = py3Dmol.view(width=800, height=800)
view.setCameraParameters({'fov': 35, 'z': 100})

# top hit for any pdb file and any smiles
top_hit = df_results.sort_values("diffdock_confidence", ascending=False).iloc[0]
print("top hit:")
display(top_hit)

# add sdf
view.addModel(open(top_hit.sdf_file).read(), "sdf")
view.setStyle({"model": 0}, {'stick':{"color":"#ff0000"}})
view.setViewStyle({"model": 0}, {'style':'outline','color':'black','width':0.1})
view.zoomTo()

# add pdb
view.addModel(open(top_hit.pdb_file).read(), "pdb")
view.setStyle({"model": 1}, {"cartoon":{"color":"spectrum"}})
view.setStyle({"model": 1, "hetflag":True}, {'stick':{"color":"spectrum"}})

model = view.getModel()
model.setHoverable({}, True, resid_hover, unhover_func)

view


['../data_files/output/diffdock/index0_data_files-inference-pdb-6agt.pdb']


runs:   0%|          | 0/1 [00:00<?, ?it/s]==============================
*** Open Babel Warning  in Init
  Unable to open data file 'space-groups.txt'
*** Open Babel Warning  in Init
  Cannot initialize database 'space-groups.txt' which may cause further errors.
*** Open Babel Warning  in parseConectRecord
  Problems reading a CONECT record.
  According to the PDB specification,
  Atoms with serial #11983 and #15925 should be connected
  However, an atom with serial #15925 was not found.
  THIS CONECT RECORD WILL BE IGNORED.

*** Open Babel Warning  in parseConectRecord
  Problems reading a CONECT record.
  According to the PDB specification,
  Atoms with serial #11988 and #15925 should be connected
  However, an atom with serial #15925 was not found.
  THIS CONECT RECORD WILL BE IGNORED.

*** Open Babel Warning  in parseConectRecord
  Problems reading a CONECT record.
  According to the PDB specification,
  columns 7-11 should contain the serial number of an atom.
  No atom was found

top hit:


pdb_file                                 ../data_files/inference/pdb/6agt.pdb
smiles                                                               6agt.pdb
diffdock_confidence                                                      0.09
gnina_scored_affinity                                                  3.5256
gnina_minimized_affinity                                             -5.70604
sdf_file                    ../data_files/output/diffdock/index0_data_file...
Name: 0, dtype: object

You appear to be running in JupyterLab (or JavaScript failed to load for some other reason). You need to install the 3dmol extension: 
 jupyter labextension install jupyterlab_3dmol